In [ ]:
!pip -q install wordfreq nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 6.5 MB/s eta 0:00:00


In [ ]:

import os, re
import pandas as pd

TSV_PATH = "NEG-136-SIMP-BAL.tsv"

if not os.path.exists(TSV_PATH):
    from google.colab import files
    uploaded = files.upload()
    TSV_PATH = next(iter(uploaded))

df = pd.read_csv(TSV_PATH, sep="\t")
print("Loaded:", TSV_PATH, df.shape)
df.head()

Saving NEG-136-SIMP-BAL.tsv.tsv to NEG-136-SIMP-BAL.tsv.tsv
Loaded: NEG-136-SIMP-BAL.tsv.tsv (70, 5)


,item,context_aff,context_neg,target_aff,target_neg
0,1,A barn is (a|an),A barn is not (a|an),building,fruit
1,2,A banana is (a|an),A banana is not (a|an),fruit,flower
2,3,A hotel is (a|an),A hotel is not (a|an),building,fish
3,4,A moth is (a|an),A moth is not (a|an),insect,bird
4,5,A chisel is (a|an),A chisel is not (a|an),tool,flower


In [ ]:

df["hyponym"]  = df["context_aff"].str.extract(r"^(?:A|An)\s+(.+?)\s+is\b", flags=re.I)[0].str.lower()
df["category"] = df["target_aff"]

missing = df[df.hyponym.isna()]
assert missing.empty, f"Could not read the hyponym in these rows:\n{missing}"
print(f"{len(df)} hyponyms extracted")

70 hyponyms extracted


In [ ]:

from wordfreq import zipf_frequency

def band(z):
    if z < 3:
        return "low (rare)"
    return "mid" if z < 4 else "high"

df["zipf"] = df["hyponym"].apply(lambda w: round(zipf_frequency(w, "en"), 2))
df["band"] = df["zipf"].apply(band)


unseen = df[df.zipf == 0]
if len(unseen):
    print("Not found in wordfreq (check spelling):", list(unseen.hyponym))

In [ ]:

try:
    import nltk
    nltk.download("wordnet", quiet=True)
    from nltk.corpus import wordnet as wn
    df["noun_senses"] = df["hyponym"].apply(lambda w: len(wn.synsets(w, pos="n")))
    print("WordNet sense counts added.")
except Exception as e:
    print("WordNet not available, skipping sense counts:", e)

WordNet sense counts added.


In [ ]:

cols = [c for c in ["item", "hyponym", "category", "zipf", "band", "noun_senses"] if c in df.columns]
freq_table = df.sort_values("zipf")[cols].reset_index(drop=True)

print(f"Median Zipf: {df.zipf.median():.2f}   Mean Zipf: {df.zipf.mean():.2f}")
print("Items per band:", df.band.value_counts().to_dict())
pd.set_option("display.max_rows", 200)
freq_table

Median Zipf: 3.78   Mean Zipf: 3.87
Items per band: {'mid': 38, 'high': 26, 'low (rare)': 6}


,item,hyponym,category,zipf,band,noun_senses
0,26,trowel,tool,2.45,low (rare),1
1,59,ladybug,insect,2.67,low (rare),1
2,68,carnation,flower,2.70,low (rare),2
3,20,radish,vegetable,2.78,low (rare),5
4,54,turnip,vegetable,2.84,low (rare),2
5,5,chisel,tool,2.96,low (rare),1
6,52,mantis,insect,3.11,mid,1
7,66,screwdriver,tool,3.13,mid,2
8,32,carp,fish,3.18,mid,2
9,14,tulip,flower,3.18,mid,1


In [ ]:

rare = freq_table[freq_table.zipf < 3]
print(f"{len(rare)} rare hyponyms in the dataset")
rare

6 rare hyponyms in the dataset


,item,hyponym,category,zipf,band,noun_senses
0,26,trowel,tool,2.45,low (rare),1
1,59,ladybug,insect,2.67,low (rare),1
2,68,carnation,flower,2.70,low (rare),2
3,20,radish,vegetable,2.78,low (rare),5
4,54,turnip,vegetable,2.84,low (rare),2
5,5,chisel,tool,2.96,low (rare),1


In [ ]:

per_cat = (df.groupby("category")
             .agg(n=("hyponym", "size"),
                  mean_zipf=("zipf", "mean"),
                  min_zipf=("zipf", "min"),
                  max_zipf=("zipf", "max"),
                  n_rare=("zipf", lambda z: int((z < 3).sum())),
                  rarest=("hyponym", lambda s: s.loc[df.loc[s.index, "zipf"].idxmin()]))
             .round(2)
             .sort_values("mean_zipf"))
per_cat

,n,mean_zipf,min_zipf,max_zipf,n_rare,rarest
category,,,,,,
tool,7,3.38,2.45,4.16,2,trowel
vegetable,7,3.43,2.78,4.11,2,radish
insect,7,3.44,2.67,4.09,1,ladybug
flower,7,3.58,2.70,4.82,1,carnation
fish,7,3.73,3.18,4.10,0,carp
tree,7,3.78,3.36,4.18,0,elm
bird,7,3.81,3.44,4.33,0,peacock
fruit,7,3.91,3.47,4.76,0,pear
vehicle,7,4.65,3.78,5.45,0,tractor


In [ ]:

freq_table.to_csv("hyponym_frequencies.csv", index=False)
per_cat.to_csv("category_frequencies.csv")
try:
    from google.colab import files
    files.download("hyponym_frequencies.csv")
    files.download("category_frequencies.csv")
except ImportError:
    print("Saved hyponym_frequencies.csv and category_frequencies.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>